In [215]:
# %pip install torch

In [216]:
import torch
import torch.nn as nn   

In [217]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.rrn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    def forward(self, x, hidden):
        out, hidden = self.rrn(x, hidden)
        out = self.fc(out)
        return out, hidden
    def init_hidden(self, batch_size):
        return torch.zeros(1, batch_size, self.hidden_size)

In [6]:
corpus = "hello world this is a recurrent neural network example"
chars = sorted(list(set(corpus)))
print(chars)

[' ', 'a', 'c', 'd', 'e', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'w', 'x']


In [218]:
char_to_idx = {ch: idx for idx, ch in enumerate(chars)}
idx_to_char = {idx: ch for ch, idx in char_to_idx.items()}

input_seq = torch.tensor([char_to_idx[ch] for ch in sequence[:-1]], dtype=torch.long)
target_seq = torch.tensor([char_to_idx[ch] for ch in sequence[1:]], dtype=torch.long)

In [219]:
def one_hot_encode(sequence, num_classes):
    return torch.eye(num_classes)[sequence]

In [220]:
input_seq_onehot = one_hot_encode(input_seq, len(char_to_idx)).unsqueeze(0)

In [221]:
input_size = len(char_to_idx)
hidden_size = 10
output_size = len(char_to_idx)
learning_rate = 0.01
num_epochs = 100

model = SimpleRNN(input_size, hidden_size, output_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for epoch in range(num_epochs):
    hidden = model.init_hidden(batch_size=1)
    output, hidden = model(input_seq_onehot, hidden)
    loss = criterion(output.view(-1, output_size), target_seq.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [10/100], Loss: 1.5331
Epoch [20/100], Loss: 0.9642
Epoch [30/100], Loss: 0.5548
Epoch [40/100], Loss: 0.2846
Epoch [50/100], Loss: 0.1476
Epoch [60/100], Loss: 0.0845
Epoch [70/100], Loss: 0.0545
Epoch [80/100], Loss: 0.0391
Epoch [90/100], Loss: 0.0301
Epoch [100/100], Loss: 0.0244


In [222]:
test_seq = "ma"
test_input = torch.tensor([char_to_idx[ch] for ch in test_seq], dtype=torch.long)
test_input_onehot = one_hot_encode(test_input, len(char_to_idx)).unsqueeze(0)

with torch.no_grad():
    hidden = model.init_hidden(batch_size=1)
    output, _ = model(test_input_onehot, hidden)
    predicted_indices = torch.argmax(output, dim=2)
    predicted_chars = [idx_to_char[idx.item()] for idx in predicted_indices.squeeze()]
    print(f"Do you mean {test_seq}{predicted_chars[0]}{predicted_chars[1]} or {test_seq}{predicted_chars[1]}{predicted_chars[0]} ?")

Do you mean mael or male ?
